In [18]:
import pandas as pd
import time
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [19]:
def split_scaler(indep_x,dep_y):
    x_train,x_test,y_train,y_test=train_test_split(indep_x,dep_y,test_size=0.25,random_state=0)
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return x_train,x_test,y_train,y_test

In [20]:
def r2_prediction(regressor,x_test,y_test):
    y_pred=regressor.predict(x_test)
    from sklearn.metrics import r2_score
    r2=r2_score(y_test,y_pred)
    return r2

In [21]:
def linear(x_train,y_train,x_test):
    from sklearn.linear_model import LinearRegression
    regressor=LinearRegression()
    regressor.fit(x_train,y_train)
    r2=r2_prediction(regressor,x_test,y_test)
    return r2

In [22]:
def svm_linear(x_train,y_train,x_test):
    from sklearn.svm import SVR
    regressor=SVR(kernel='linear')
    regressor.fit(x_train,y_train)
    r2=r2_prediction(regressor,x_test,y_test)
    return r2

In [23]:
def svm_nonlinear(x_train,y_train,x_test):
    from sklearn.svm import SVR
    regressor=SVR(kernel='rbf')
    regressor.fit(x_train,y_train)
    r2=r2_prediction(regressor,x_test,y_test)
    return r2

In [24]:
def decision_tree(x_train,y_train,x_test):
    from sklearn.tree import DecisionTreeRegressor
    regressor=DecisionTreeRegressor(random_state=0)
    regressor.fit(x_train,y_train)
    r2=r2_prediction(regressor,x_test,y_test)
    return r2

In [25]:
def random_forest(x_train,y_train,x_test):
    from sklearn.ensemble import RandomForestRegressor
    regressor=RandomForestRegressor(n_estimators=10,random_state=0)
    regressor.fit(x_train,y_train)
    r2=r2_prediction(regressor,x_test,y_test)
    return r2

In [26]:
def rfeFeature(indep_x,dep_y,n):
        rfelist=[]
        from sklearn.linear_model import LinearRegression
        lin = LinearRegression()
        from sklearn.svm import SVR
        SVRl = SVR(kernel = 'linear')
        from sklearn.svm import SVR
        #SVRnl = SVR(kernel = 'rbf')
        from sklearn.tree import DecisionTreeRegressor
        dec = DecisionTreeRegressor(random_state = 0)
        from sklearn.ensemble import RandomForestRegressor
        rf = RandomForestRegressor(n_estimators = 10, random_state = 0)
       
    #use ML algorithm to select the features
        rfemodellist=[lin,SVRl,dec,rf] 
        for i in   rfemodellist:
            print(i)
            log_rfe = RFE(estimator=i,n_features_to_select=n)
            log_fit = log_rfe.fit(indep_x, dep_y)
            log_rfe_feature=log_fit.transform(indep_x)
            rfelist.append(log_rfe_feature)
        return rfelist

In [35]:
def rfe_regression(acclin,accsvml,accdes,accrf):
      dataframe=pd.DataFrame(index=['linear','SVMl','decision','random'],columns=['Lin','SVMl','DT','RF'])
      for number,idex in enumerate(dataframe.index):  #enumerate return two values index,value
          dataframe['Lin'][idex]=acclin[number]
          dataframe['SVMl'][idex]=accsvml[number]
          dataframe['DT'][idex]=accdes[number]
          dataframe['RF'][idex]=accrf[number]
      return dataframe

In [36]:
dataset=pd.read_csv('prep.csv',index_col=None)
df=dataset
df=pd.get_dummies(df,dtype=int,drop_first=True)
indep_x=df.drop('classification_yes',axis=1)
dep_y=df['classification_yes']


In [44]:
rfelist=rfeFeature(indep_x,dep_y,9)
acclin=[]
accsvml=[]
accsvmnl=[]
accdes=[]
accrf=[]
for i in rfelist:
    x_train,x_test,y_train,y_test=split_scaler(i,dep_y)
    r2_lin=linear(x_train,y_train,x_test)
    acclin.append(r2_lin)
    r2_svml=svm_linear(x_train,y_train,x_test)
    accsvml.append(r2_svml)
    r2_svmnl=svm_nonlinear(x_train,y_train,x_test)
    accsvmnl.append(r2_svmnl)
    r2_des=decision_tree(x_train,y_train,x_test)
    accdes.append(r2_des)
    r2_rf=random_forest(x_train,y_train,x_test)
    accrf.append(r2_rf)
result=rfe_regression(acclin,accsvml,accdes,accrf)

LinearRegression()
SVR(kernel='linear')
DecisionTreeRegressor(random_state=0)
RandomForestRegressor(n_estimators=10, random_state=0)


C:\Users\Sathish\AppData\Local\Temp\ipykernel_3852\4157567126.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  dataframe['Lin'][idex]=acclin[number]
C:\Users\Sathish\AppData\Local\Temp\ipykernel_3852\4157567126.py:5: FutureWarning: Chaine

In [41]:
result
#6

,Lin,SVMl,DT,RF
linear,0.624738,0.456874,0.81723,0.814741
SVMl,0.610294,0.530043,0.806415,0.807916
decision,0.697365,0.665248,0.782986,0.829427
random,0.705126,0.670093,0.839675,0.875221


In [43]:
result
#5

,Lin,SVMl,DT,RF
linear,0.620124,0.457136,0.77924,0.780135
SVMl,0.604508,0.456871,0.776474,0.776745
decision,0.674403,0.628206,0.696181,0.815538
random,0.686361,0.643365,0.836806,0.845303


In [45]:
result
#9

,Lin,SVMl,DT,RF
linear,0.716216,0.684977,0.968654,0.958692
SVMl,0.700809,0.682756,0.82978,0.924309
decision,0.713384,0.687787,0.782986,0.912326
random,0.71276,0.675771,0.739583,0.904948
